# From the last vector to a token: unembedding, logits, temperature, sampling (GPT-2 small, NumPy)
Weights: `tools/gpt2_small.py` (Hugging Face safetensors, cached in ~/.cache/campusx/gpt2; first run downloads about 500 MB).
1. Unembedding: logits = final vector . every row of W_E; logit vs cosine.
2. Softmax with temperature on real logits; entropy and tail mass vs T.
3. Sampling at several temperatures: what changes in the text.
4. Nucleus (top-p) sizes on two prompts.
5. Logit lens: the next-token guess after every block.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "tools"))
import gpt2_small as g

np.set_printoptions(precision=3, suppress=True)
DATA = Path("data"); DATA.mkdir(exist_ok=True)
W = g.load()
WE = W["wte.weight"]
show = lambda i: g.decode([int(i)])

## 1. Unembedding: one dot product per vocabulary token

In [ ]:
prompt = "Steve Jobs was the founder of"
ids = g.encode(prompt)
out = g.forward(ids, W)
h = out["final"][-1]                                   # after ln_f
logits = WE @ h                                        # 50,257 dot products
assert np.allclose(logits, out["logits"][-1], atol=1e-3)
norm_w = np.linalg.norm(WE, axis=1)
cos = logits / (np.linalg.norm(h) * norm_w)
top = np.argsort(-logits)[:10]
p = g.softmax(logits)
t1 = pd.DataFrame({"token": [show(i) for i in top], "logit": logits[top], "probability": p[top],
                   "cosine": cos[top], "row_length": norm_w[top]})
print("final vector length:", np.linalg.norm(h).round(1), " rows of W_E: median length", np.median(norm_w).round(2))
print(t1.round(3).to_string(index=False))
print("correlation(logit, cosine) over all tokens:", np.corrcoef(logits, cos)[0, 1].round(3))
print("rank of ' Apple' by logit:", int((logits > logits[g.encode(' Apple')[0]]).sum()) + 1,
      " by cosine:", int((cos > cos[g.encode(' Apple')[0]]).sum()) + 1)
print("logit range:", logits.min().round(1), "to", logits.max().round(1), "; mean", logits.mean().round(1))
# The rows of W_E share one direction: their mean row. Its part of every logit is the same number for all tokens.
m = WE.mean(0)
print("cosine of each row with the mean row, average:", np.mean(WE @ m / norm_w / np.linalg.norm(m)).round(2))
print("h . mean row:", float(h @ m), " mean logit:", logits.mean())
Cn = WE - m
ccos = (Cn @ h) / (np.linalg.norm(Cn, axis=1) * np.linalg.norm(h))
print("correlation(logit, cosine with centred rows):", np.corrcoef(logits, ccos)[0, 1].round(3))
print("top 6 by centred cosine:", [show(i) for i in np.argsort(-ccos)[:6]])
t1["centred_cosine"] = ccos[top]
print(t1.round(3).to_string(index=False))
# softmax ignores a constant added to all logits
assert np.allclose(g.softmax(logits), g.softmax(logits - h @ m))
rng = np.random.default_rng(0)
keep = np.unique(np.concatenate([top, np.argsort(-logits)[:300], rng.choice(len(logits), 3000, replace=False)]))
pd.DataFrame({"token": [show(i) for i in keep], "logit": logits[keep], "cosine": cos[keep], "centred_cosine": ccos[keep],
              "top10": np.isin(keep, top)}).to_csv(DATA / "logit_vs_cosine.csv", index=False)

## 2. Softmax with temperature
p_k = exp(z_k / T) / sum_j exp(z_j / T) (Holtzman et al. 2020, eq. 4). Prompt: "Once upon a time there was a" (Sanderson 2024, Ch 5 uses it with GPT-3).

In [ ]:
story = "Once upon a time there was a"
zs = g.forward(g.encode(story), W)["logits"][-1]
order = np.argsort(-zs)
temps = np.round(np.concatenate([np.arange(0.05, 1.0, 0.05), np.arange(1.0, 3.01, 0.1)]), 2)
rows = []
for T in temps:
    q = g.softmax(zs, T)
    H = -(q[q > 0] * np.log2(q[q > 0])).sum()
    rows.append(dict(T=T, entropy_bits=H, top1=q[order[0]], top10_mass=q[order[:10]].sum(),
                     outside_top10=1 - q[order[:10]].sum(), effective_choices=2 ** H))
temp = pd.DataFrame(rows)
temp.to_csv(DATA / "temperature_sweep.csv", index=False)
print("uniform over 50,257 tokens:", np.log2(50257).round(2), "bits")
print(temp[temp["T"].isin([0.05, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0])].round(4).to_string(index=False))
# the bars for the animation: top-10 tokens, their logits
pd.DataFrame({"token": [show(i) for i in order[:10]], "logit": zs[order[:10]]}).to_csv(DATA / "story_top10.csv", index=False)
np.save(DATA / "story_logits.npy", zs.astype(np.float32))      # all 50,257 logits, for the temperature animation
print(pd.read_csv(DATA / "story_top10.csv").round(2).to_string(index=False))

Check of the two limits on the real logits: T -> 0 gives all the weight to the top token, large T approaches uniform.

In [ ]:
for T in (0.01, 1.0, 100.0, 1e4):
    q = g.softmax(zs, T)
    print(f"T = {T:>8}: top token {q.max():.4f}; smallest {q.min():.2e}; uniform would be {1/50257:.2e}")

## 3. Sampling at several temperatures
10 continuations of 12 tokens per temperature (seeds 0-9). T = 0 is greedy (always the top token).

In [ ]:
samples = []
for T in (0.0, 0.3, 0.7, 1.0, 1.5):
    for s in range(10 if T > 0 else 1):
        ids_s = g.generate(g.encode(story), W, n=12, T=T, seed=s)
        samples.append(dict(T=T, seed=s, text=g.decode(ids_s[len(g.encode(story)):])))
samp = pd.DataFrame(samples)
samp.to_csv(DATA / "samples.csv", index=False)
for T, grp in samp.groupby("T"):
    print(f"\nT = {T}: {grp.text.nunique()} distinct of {len(grp)}")
    for t in grp.text.head(3):
        print("   ", repr(t))

Longer greedy run: repetition (the published example of von Platen 2020 also repeats).

In [ ]:
greedy = g.decode(g.generate(g.encode(story), W, n=40))
print(repr(greedy))

Mean log-probability (under the model, T = 1) of the sampled tokens, per temperature: low T picks likely tokens, high T unlikely ones.

In [ ]:
def mean_logprob(text_cont):
    ids_full = g.encode(story + text_cont)
    n0 = len(g.encode(story))
    lp = np.log(g.softmax(g.forward(ids_full, W)["logits"][:-1]))
    return float(np.mean([lp[i - 1, ids_full[i]] for i in range(n0, len(ids_full))]))

samp["mean_logprob"] = [mean_logprob(t) for t in samp.text]
samp.to_csv(DATA / "samples.csv", index=False)
print(samp.groupby("T").mean_logprob.mean().round(2))

## 4. Nucleus (top-p) sampling: how many tokens hold 90 percent of the probability?

In [ ]:
for pr in (prompt, story):
    q = np.sort(g.softmax(g.forward(g.encode(pr), W)["logits"][-1]))[::-1]
    print(f"{pr!r}: top-1 {q[0]:.3f}; tokens needed for 90 percent: {int(np.searchsorted(np.cumsum(q), 0.9)) + 1}")

## 5. Logit lens: unembed the residual stream after every block (through ln_f), nostalgebraist (2020)

In [ ]:
L = g.logit_lens(out, W)                               # (13, T, vocab)
PL = g.softmax(L)
apple = g.encode(" Apple")[0]
lens = []
for l in range(13):
    q = PL[l, -1]; tp = np.argsort(-q)[:8]
    lens.append(dict(layer=l, apple_prob=q[apple], apple_rank=int((q > q[apple]).sum()) + 1,
                     **{f"tok{j}": show(t) for j, t in enumerate(tp)}, **{f"p{j}": q[t] for j, t in enumerate(tp)}))
lens = pd.DataFrame(lens)
lens.to_csv(DATA / "logit_lens_last.csv", index=False)
print(lens[["layer", "apple_rank", "apple_prob", "tok0", "p0", "tok1", "tok2"]].round(3).to_string(index=False))

The top guess at every position and every layer (the grid of nostalgebraist 2020).

In [ ]:
grid = [dict(layer=l, position=i, input=show(ids[i]), guess=show(PL[l, i].argmax()), prob=PL[l, i].max(),
             final_guess=show(PL[12, i].argmax())) for l in range(13) for i in range(len(ids))]
grid = pd.DataFrame(grid)
grid.to_csv(DATA / "logit_lens_grid.csv", index=False)
print(grid.pivot(index="layer", columns="position", values="guess").to_string())
same = (grid.guess == grid.final_guess).groupby(grid.layer).mean()
print("share of positions whose top guess already equals the final one, by layer:\n", same.round(2).to_string())

A second fact prompt, to see whether the pattern repeats.

In [ ]:
for pr, tgt in (("The Statue of Liberty is in New", " York"), ("Tiger Woods plays the sport of", " golf"),
                ("Michael Jordan plays the sport of", " basketball")):
    o2 = g.forward(g.encode(pr), W); P2 = g.softmax(g.logit_lens(o2, W)[:, -1]); t = g.encode(tgt)[0]
    ranks = [int((P2[l] > P2[l, t]).sum()) + 1 for l in range(13)]
    print(f"{pr!r} -> {tgt!r}: rank by layer {ranks}; prob at layers 8-12 {P2[8:, t].round(3)}")